## LLMOps - Langfuse 설치

## 디버깅의 어려움
- 전통적인 소프트웨어는 스택 트레이스는 로그를 통해 문제를 추적할 수 있음.
- 하지만 LLM 어플리케이션에서는 다음과 같은 어려움 존재

### 디버깅의 어려움  
**문제상황**   
- 사용자가 "답변이 이상해요"라고 보고
- 어떤 프롬프트가 사용되었는지 알 수 없음.
- 모델이 어떤 컨텍스트를 받았는지 불명확
- 중간 단계의 에이전트 동작을 추적할 수 없음.

**필요한 정보**
- 전체 프롬프트(시스템메시지+사용자입력+컨텍스트)
- 모델 파라미터
- 응답생성시간
- 토큰 사용량 및 비용
- 에이전트의 도구 호출 순서

## 비용관리의 복잡성
- LLM API 호출은 토큰 단위로 과금되며, 비용을 예측하기 어려움

## 품질 평가의 주관성
- 전통적인 소프트웨어는 단위 테스트로 검증이 가능함.
- LLM의 출력은 정답이 주관적이며, 명확하게 답이 정해져있지 않음.

## 프로덕션 모니터링의 필요성
- 개발환경에서 잘 작동하던 LLM 애플리케이션이 프로덕션에서 문제를 일으킬 수 있음.   

**모니터링 해야 할 지표**
- 응답 품질(사용자 피드백, 평가 점수)
- 응답시간(레이턴시)
- 비용(일일/월간 토큰 사용량)
- 에러율(API실패, 타임아웃)
- 사용패턴(어떤 기능이 많이 사용되는가)

## LLMOps의 등장
- 이러한 도전과제들을 해결하기 위해 LLMOps(Large Language Model Operations)라는 새로운 분야 등장

**LLMOps의 핵심 요소**
- 관찰성(Observability): 모든 LLM 호출을 추적하고 로깅
- 평가(Evaluation): 자동화된 품질 평가 프레임워크
- 모니터링(Monitoring): 실시간 성능 및 비용 대시보드
- 실험 관리(Experimentation): 프롬프트 버전 관리 및 A/B 테스트
- 데이터관리(Data Management): 사용자 입력/출력 데이터 수집 및 분석

```{markdown}
MLOps 와의 차이점
MLOps는 전통적인 머신러닝 모델의 학습/배포/모니터링에 집중합니다. LLMOps는 이미 학습된 대형 모델을 프롬프트와 컨텍스트로 제어하는 데 초점을 맞춥니다. 따라서 프롬프트 엔지니어링, 토큰 비용 관리, 에이전트 추적과 같은 고유한 도구가 필요합니다.
```

## Langfuse 가 해결하는 문제
- Langfuse는 위에서 언급한 도전 과제를 해결하기 위한 오픈소스 LLMOps 플랫폼임.

### Langfuse의 핵심 기능
- 트레이싱: 모든 LLM 호출과 에이전트 동작을 자동으로 기록
- 비용추적: 토큰 사용량과 비용을 실시간으로 집계
- 프롬프트 관리: 프롬프트를 버전별로 관리하고 배포
- 평가: 자동화된 평가 파이프라인 구축
- 대쉬보드: 사용 패턴과 성능 지표를 UI로 시각화

## Langfuse 핵심 개념
- Langfuse의 데이터모델은 트레이스(Trace)와 관찰(Observation)으로 구성됨.

### Trace
- 하나의 사용자 요청에 대한 전체 실행 흐름

### 관찰(Observation)
- 트레이스 내부의 개별 작업 단위. 3가지 타입으로 구성됨

1. Span: 일반적인 작업(예: 데이터베이스 쿼리, 함수 실행 등등)
2. Generation: LLM 호출(입력, 출력, 모델, 토큰 수 포함)
3. Event: 특정 시점의 이벤트(로그, 메트릭 등)


In [ ]:
from langfuse.openai import openai
from langfuse import observe

@observe()  # 이 함수는 하나의 Span이 됨
def retrieve_documents(query):
    # 문서 검색 로직
    return documents

@observe()  # 이 함수도 Span이 됨
def generate_answer(documents, query):
    # LLM 호출 → Generation으로 자동 기록
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"Documents: {documents}\n\nQuestion: {query}"}
        ]
    )
    return response.choices[0].message.content

# 전체 실행 흐름
@observe()  # 최상위 트레이스
def rag_pipeline(user_query):
    docs = retrieve_documents(user_query)  # Span
    answer = generate_answer(docs, user_query)  # Span + Generation
    return answer


### 계층 구조
```
Trace: rag_pipeline
├── Span: retrieve_documents
│   └── Event: found 5 documents
└── Span: generate_answer
    └── Generation: gpt-4o call
        ├── Input tokens: 150
        ├── Output tokens: 80
        └── Cost: $0.0032
```

### Langfuse 아키텍쳐 구성
1. langfuse SDK
2. Langfuse server: 트레이스 데이터를 저장하고 대쉬보드를 제공하는 백엔드 서버   
   - API 엔드포인트: SDK에서 트레이스를 전송받음
   - 데이터베이스: PostgreSQL에 트레이스 저장
   - 웹대쉬보드: React 기반 UI 제공

3. Langfuse UI
- 웹 브라우저에서 접근 가능한 대쉬보드  
   
  - 트레이스 목록 및 상세 보기
  - 비용 및 사용량 분석
  - 프롬프트 관리
  - 평가 결과 확인

## Langfuse가 제공하는 가치
### 1. 개발 단계

- 프롬프트 실험 결과를 즉시 확인
- 에이전트 동작을 시각적으로 디버깅
- 로컬 개발 시 트레이스 확인 가능

### 2. 스테이징 단계

- 통합 테스트 결과를 트레이스로 기록
- 성능 병목 지점 파악
- 비용 추정

### 3. 프로덕션 단계

- 실시간 모니터링 대시보드
- 사용자 피드백과 트레이스 연결
- 이상 감지 및 알림
- 비용 초과 방지

## Langfuse Cloud VS 셀프 호스팅

### Langfuse Cloud
- 설치 불필요
- 즉시 사용가능
- EU/US 리전 선택 가능
- 프로토타입이나 중소규모 프로젝트에 적합

### 셀프 호스팅
- 데이터를 자체 서버에 보관
- 커스터마이징 가능
- 규제 요구사항이 있거나 대규모 트래픽을 처리하는 기업에 적합

## Setting

### Hello langfuse

In [ ]:
from langfuse.openai import openai
from dotenv import load_dotenv

load_dotenv()
response = openai.chat.completions.create(
    name = "hello-langfuse",
    model = "gpt-4o-mini",
    messages=[
        {"role": "system" , "content": "you are a helpful assistant"},
        {"role": "user" , "content": "hello langfuse"}
    ],
    metadata = {"environment": "development"}
)

print(response.choices[0].message.content)

### 함수 데코레이터 사용
- python에서는 ```@observe``` 데코레이터를 사용하여 모든 함수를 자동으로 트레이스 할 수 있음.

In [ ]:
# advanced_example.py
from langfuse import observe, get_client
from langfuse.openai import openai
from dotenv import load_dotenv

load_dotenv()

@observe()  # 이 함수가 Span으로 기록됨
def retrieve_context(query: str) -> str:
    """사용자 질문에 대한 컨텍스트를 검색합니다."""
    # 실제로는 벡터 DB 검색 등을 수행
    return f"Langfuse는 LLM 애플리케이션을 위한 관찰성 플랫폼입니다."

@observe()  # 이 함수도 Span으로 기록됨
def generate_answer(query: str, context: str) -> str:
    """컨텍스트를 기반으로 답변을 생성합니다."""
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Use this context: {context}"},
            {"role": "user", "content": query}
        ]
    )
    return response.choices[0].message.content

@observe()  # 최상위 함수가 Trace가 됨
def rag_pipeline(user_query: str) -> str:
    """RAG 파이프라인 전체를 실행합니다."""
    context = retrieve_context(user_query)
    answer = generate_answer(user_query, context)
    return answer

# 실행
if __name__ == "__main__":
    result = rag_pipeline("Langfuse란 무엇인가요?")
    print(result)

    # 짧은 애플리케이션에서는 flush 필요
    langfuse = get_client()
    # 비동기 전송 완료 대기 - 스크립트 종료 전 데이터 누락 방지
    langfuse.flush()


* 위의 Trace 계층 구조

```
Trace: rag_pipeline
├── Span: retrieve_context
└── Span: generate_answer
    └── Generation: gpt-4o-mini call
```

# 2. Langfuse Tracing 마스터하기

## 2.1 Trace
- 하나의 요청이 시스템을 통과하는 전체 여정을 나타냄.



## Tracing 실습

In [1]:
from langfuse import observe, get_client

langfuse = get_client()

@observe(name="data-model-demo")
def demo():
    print(f"Trace ID: {langfuse.get_current_trace_id()}")
    print(f"Observation ID: {langfuse.get_current_observation_id()}")

    nested_operation()
    return "완료"

@observe(as_type="generation", name="llm-call")
def nested_operation():
    print(f"  자식 Observation ID: {langfuse.get_current_observation_id()}")
    print(f"  같은 Trace ID: {langfuse.get_current_trace_id()}")

demo()
langfuse.flush()


Trace ID: c7dcc548951b9486279212a174b20579
Observation ID: e6dbac0b4a1673d3
  자식 Observation ID: 3474857258a39a82
  같은 Trace ID: c7dcc548951b9486279212a174b20579


## 2.2 @observe 데코레이터

In [ ]:
from langfuse import observe

@observe()
def my_function(input_data):
    # 자동으로 캡처되는 것들:
    # - 함수 이름 (span 이름)
    # - 입력 인자
    # - 반환 값
    # - 실행 시간
    # - 에러 (발생 시)
    return process(input_data)


### 데코레이터 파라미터

name : str -> 커스텀 이름(기본 함수명)   
as_type: str -> 관찰 타입 지정: "generation" (LLM 호출), "agent" (에이전트), "tool" (도구), "retriever" (검색), "guardrail" (가드레일)   
capture_intent : bool -> 입력 캡쳐 여부   
capture_output : bool -> 출력 캡쳐 여부   

In [ ]:
from langfuse import observe

@observe(
    name="customer-query-processor",
    as_type="generation",
    capture_input=True,
    capture_output=True
)
def process_query(query: str) -> str:
    return llm.generate(query)


### Generation 타입 활용

- LLM 호출에는 ```as_type = "generation"``` 을 추가필드로 지정함.

In [ ]:
from langfuse.openai import openai
from langfuse import observe, get_client

client = openai.OpenAI()
langfuse = get_client()

@observe(as_type="generation")
def call_llm(prompt: str):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )

    # Generation 전용 필드 업데이트
    langfuse.update_current_generation(
        model="gpt-4o",
        usage_details={
            "input": response.usage.prompt_tokens,
            "output": response.usage.completion_tokens
        },
        model_parameters={"temperature": 0.7}
    )

    return response.choices[0].message.content


### 자동 중첩
- 함수 호출 계층이 자동으로 트레이싱 구조에 반영됨.

In [ ]:
from langfuse import observe

@observe()
def main_workflow(user_input: str):
    # 1단계: 전처리
    processed = preprocess(user_input)

    # 2단계: LLM 호출
    response = generate_response(processed)

    # 3단계: 후처리
    final = postprocess(response)

    return final

@observe()
def preprocess(text: str):
    return text.strip().lower()

@observe(as_type="generation")
def generate_response(text: str):
    # LLM 호출 로직
    return "응답"

@observe()
def postprocess(response: str):
    return {"result": response, "processed": True}

# 실행하면 다음 구조의 트레이스 생성:
# main_workflow
# ├── preprocess
# ├── generate_response (generation)
# └── postprocess


### 비동기 함수도 동일하게 지원됨

In [ ]:
from langfuse import observe
import asyncio

@observe()
async def async_workflow():
    results = await asyncio.gather(
        fetch_data(),
        process_data()
    )
    return results

@observe()
async def fetch_data():
    await asyncio.sleep(0.1)
    return "data"

@observe()
async def process_data():
    await asyncio.sleep(0.1)
    return "processed"


In [1]:
from langfuse import get_client

langfuse = get_client()

with langfuse.start_as_current_observation(
    as_type="span",
    name="outer-operation"
) as outer:
    print(f"Outer ID: {outer.id}")

    with langfuse.start_as_current_observation(
        as_type="span",
        name="inner-operation"
    ) as inner:
        print(f"Inner ID: {inner.id}")
        # inner는 자동으로 outer의 자식

    with langfuse.start_as_current_observation(
        as_type="generation",
        name="llm-call",
        model="gpt-4o"
    ) as gen:
        # gen도 outer의 자식
        response = "LLM 응답"
        gen.update(output=response)


Outer ID: 33c6836891892e26
Inner ID: 2fa028b4fd899ecb


In [ ]:
from langfuse import observe, get_client
import time

langfuse = get_client()

@observe()
def order_processing(order_id: str):
    """주문 처리 워크플로우"""

    # 트레이스 메타데이터 설정
    langfuse.update_current_trace(
        name=f"order-{order_id}",
        metadata={"order_id": order_id}
    )

    # 1. 재고 확인
    with langfuse.start_as_current_observation(
        as_type="span",
        name="inventory-check"
    ) as inv:
        time.sleep(0.1)  # 시뮬레이션
        inventory = {"available": True, "quantity": 10}
        inv.update(output=inventory)

    # 2. 결제 처리
    with langfuse.start_as_current_observation(
        as_type="span",
        name="payment-processing"
    ) as pay:
        time.sleep(0.2)
        payment = {"status": "approved", "amount": 99.99}
        pay.update(
            output=payment,
            metadata={"gateway": "stripe"}
        )

    # 3. AI 추천 생성
    with langfuse.start_as_current_observation(
        as_type="generation",
        name="recommendation-generation",
        model="gpt-4o-mini"
    ) as rec:
        rec.update(
            input={"order_id": order_id},
            output=["추천 상품 A", "추천 상품 B"],
            usage_details={"input": 50, "output": 30}
        )

    return {
        "order_id": order_id,
        "status": "completed",
        "recommendations": ["추천 상품 A", "추천 상품 B"]
    }

# 실행
result = order_processing("ORD-12345")
langfuse.flush()
